# MATH GR5360 Final Project — Two-Market Trend-Following Story

This notebook takes the diagnostics narrative and carries it into the trend-following implementation. It uses the same reusable story workflow to run `TY` first and then `BTC`, while keeping the strategy layer explicitly TF-only.

In [ ]:
MARKETS_TO_RUN = ['TY', 'BTC']
QUICK_TEST = True
RUN_EXTENDED_SURFACE = False

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
warnings.filterwarnings('ignore')

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'mafn_engine').exists() else CWD.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from mafn_engine import (
    COLUMBIA_CORE,
    COLUMBIA_NAVY,
    COLUMBIA_WARM,
    apply_columbia_theme,
    build_pair_story,
)

apply_columbia_theme()
DATA_DIR = str(PROJECT_ROOT / 'data')

pair_story = build_pair_story(
    MARKETS_TO_RUN,
    data_dir=DATA_DIR,
    quick=QUICK_TEST,
    walkforward_mode='tf',
    include_walkforward=True,
    include_surface=RUN_EXTENDED_SURFACE,
    verbose=False,
)
stories = pair_story['stories']
strategy_story_df = pair_story['strategy_df'].copy()

display(Markdown('## Cross-Market TF Summary'))
display(strategy_story_df.round(4))

In [ ]:
for ticker in pair_story['tickers']:
    story = stories[ticker]
    cfg = story['tf_config']
    oos = story['oos_metrics']
    full = story['full_sample_metrics']
    bullets = [
        story['narrative_lines'][0],
        f"Story TF configuration: L={int(cfg['L'])} bars, S={float(cfg['S']):.3f}.",
        f"OOS result: Profit={oos['Total Profit']:+,.2f}, MaxDD={oos['Max Drawdown $']:,.2f}, RoA={oos['Return on Account']:+.3f}." if oos else 'OOS result: not available.',
        f"Full-sample TF result: Profit={full['Total Profit']:+,.2f}, MaxDD={full['Max Drawdown $']:,.2f}, RoA={full['Return on Account']:+.3f}.",
    ]
    bullet_text = '\n'.join(f'- {line}' for line in bullets)
    display(Markdown(f"### {ticker} — TF Implementation View\n{bullet_text}"))


In [ ]:
fig, axes = plt.subplots(len(pair_story['tickers']), 2, figsize=(16, 5 * len(pair_story['tickers'])))
if len(pair_story['tickers']) == 1:
    axes = np.asarray([axes])

for row, ticker in enumerate(pair_story['tickers']):
    story = stories[ticker]
    wf_bundle = story['walkforward']
    params_df = wf_bundle['params'] if wf_bundle is not None else pd.DataFrame()
    equity_df = wf_bundle['equity'] if wf_bundle is not None else pd.DataFrame()
    ax0, ax1 = axes[row]
    if len(equity_df):
        ax0.plot(equity_df.index, equity_df['OOS_Equity'], color=COLUMBIA_CORE)
        ax0.set_title(f"{ticker}: Walk-Forward OOS Equity")
        ax0.set_ylabel('equity')
    else:
        ax0.text(0.5, 0.5, 'No walk-forward equity', ha='center', va='center')
        ax0.set_title(f"{ticker}: Walk-Forward OOS Equity")
    if len(params_df) and 'L' in params_df:
        ax1.plot(params_df['Period'], params_df['L'], color=COLUMBIA_WARM, marker='o')
        ax1.axhline(story['tf_config']['L'], color=COLUMBIA_NAVY, ls='--', lw=1.0)
        ax1.set_title(f"{ticker}: Selected TF Lookback by Period")
        ax1.set_xlabel('walk-forward period')
        ax1.set_ylabel('L (bars)')
    else:
        ax1.text(0.5, 0.5, 'No walk-forward parameter table', ha='center', va='center')
        ax1.set_title(f"{ticker}: Selected TF Lookback by Period")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(pair_story['tickers']), figsize=(8 * len(pair_story['tickers']), 4))
if len(pair_story['tickers']) == 1:
    axes = [axes]

for ax, ticker in zip(axes, pair_story['tickers']):
    story = stories[ticker]
    equity = np.asarray(story['full_sample']['Equity'], dtype=float)
    ax.plot(np.arange(len(equity)), equity, color=COLUMBIA_CORE)
    ax.set_title(f"{ticker}: Full-Sample TF Equity")
    ax.set_xlabel('bar index')
    ax.set_ylabel('equity')

plt.tight_layout()
plt.show()

In [ ]:
if RUN_EXTENDED_SURFACE:
    for ticker in pair_story['tickers']:
        story = stories[ticker]
        display(Markdown(f"### {ticker} — Extended T / tau Surface"))
        display(story['surface'])
else:
    print('Extended surface disabled. Set RUN_EXTENDED_SURFACE = True to compute it.')